# Entitle: Safety Net Benefits Navigator

Entitle helps people discover US safety net benefits in plain language using Gemma 4 structured output, reasoning, multilingual support, and document understanding.

## Problem

Tens of billions of dollars in public benefits go unclaimed because eligibility rules are confusing, application paths vary by state, and benefit notices are hard to understand. Entitle acts as a local, privacy-preserving navigation layer.

## Architecture

```text
User chat or document photo
  -> FastAPI backend
  -> Gemma 4 profile extraction / reasoning / vision
  -> Static federal + state benefits JSON
  -> Structured eligibility results and application checklist
```

In [ ]:
!pip install fastapi uvicorn httpx pydantic pydantic-settings python-multipart google-genai nest_asyncio requests pillow --quiet

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    print("Set GEMINI_API_KEY in Kaggle secrets before running the live model demo.")

os.environ["MODEL_BACKEND"] = "gemini_api"
os.environ["GEMINI_MODEL"] = "gemma-4-26b-a4b-it"
os.environ["ENABLE_THINKING"] = "true"

In [ ]:
import nest_asyncio
import sys
import threading
import time
import uvicorn
from pathlib import Path

nest_asyncio.apply()
sys.path.insert(0, str(Path.cwd() / "backend"))

def run_server():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

In [ ]:
import requests

requests.get("http://localhost:8000/health").json()

In [ ]:
scenario = "Hi, I'm a single mom with 2 kids ages 2 and 6. I work part-time and make about $1,400 a month. We live in Texas, rent an apartment, and do not have health insurance."

response = requests.post(
    "http://localhost:8000/api/chat",
    json={"message": scenario, "history": [], "language": "en"},
    timeout=180,
)
data = response.json()
print("USER:", scenario)
print("ENTITLE:", data["reply"])
print("\nPROGRAMS FOUND:")
for benefit in data.get("benefits_found") or []:
    print(f"- {benefit['program_name']}: ${benefit.get('estimated_monthly_value_usd') or 0:,.0f}/month")

In [ ]:
spanish = "Hola, somos 3 personas, gano $1500 al mes y vivimos en Texas. Tengo dos hijos."
response = requests.post(
    "http://localhost:8000/api/chat",
    json={"message": spanish, "history": [], "language": "es"},
    timeout=180,
)
print(response.json()["reply"])

In [ ]:
elderly_profile = {
    "household_size": 1,
    "adults": 1,
    "children": 0,
    "monthly_income_usd": 980,
    "state": "FL",
    "has_elderly": True,
    "has_disabled": False,
    "has_health_insurance": False,
    "housing_status": "renter",
    "utility_bills": True,
    "profile_complete": True,
}

response = requests.post(
    "http://localhost:8000/api/eligibility",
    json={"profile": elderly_profile},
    timeout=180,
)
data = response.json()
print("ELDERLY DEMO PROFILE:", elderly_profile)
print("\nTOP MATCHES:")
for benefit in data.get("results", [])[:5]:
    print(f"- {benefit['program_name']}: {benefit['reason']}")

In [ ]:
from PIL import Image, ImageDraw

notice_path = "sample_snap_renewal_notice.png"
image = Image.new("RGB", (1100, 750), "white")
draw = ImageDraw.Draw(image)
lines = [
    "SNAP Renewal Notice",
    "Case Number: 123456",
    "Your food benefits are due for renewal.",
    "Please return your renewal form by June 15, 2026.",
    "If you do not return the form, your SNAP benefits may stop.",
    "You may ask for help or appeal if you disagree with this notice.",
]
y = 70
for line in lines:
    draw.text((70, y), line, fill="black")
    y += 80
image.save(notice_path)

with open(notice_path, "rb") as f:
    response = requests.post(
        "http://localhost:8000/api/document",
        files={"file": (notice_path, f, "image/png")},
        data={"language": "en"},
        timeout=180,
    )

document = response.json()
print("DOCUMENT TYPE:", document.get("document_type"))
print("SUMMARY:", document.get("plain_language_summary"))
print("ACTION:", document.get("action_required"))
print("DEADLINE:", document.get("deadline"))

## Gemma 4 Features Used

- Structured JSON output for profile extraction and eligibility results
- Configurable thinking mode for eligibility reasoning
- Multilingual responses for users who prefer Spanish or other supported languages
- Multimodal document reading for notices, forms, denials, and renewals
- Offline-first local inference through Ollama for privacy-sensitive intake

## Disclaimer

Entitle is not a government service and does not provide legal advice. It gives estimates and application preparation help only. Official eligibility decisions are made by the relevant agencies.